# Paper 2: Spatial Biomarker Effective Sample Size Analysis
## E-MTAB-13530 — NSCLC 10x Visium

**Core claim:** Spatial autocorrelation collapses effective sample size in tissue biomarkers, creating a design-effect-driven measurement error that attenuates survival associations.

**This notebook computes:**
1. $m_i$ (spots per section)
2. $\hat{\rho}_i$ (Moran's I — spatial autocorrelation)
3. $\text{DEFF}_i = 1 + (m_i - 1)\hat{\rho}_i$ (design effect)
4. $m_{i,\text{eff}} = m_i / \text{DEFF}_i$ (effective sample size)
5. Figure 1: $m_{i,\text{eff}}$ vs $m_i$
6. Attenuation bias estimation
7. GRC correction demonstration

---
## Block 1: Install Dependencies

In [ ]:
!pip install scanpy squidpy anndata h5py scipy numpy pandas matplotlib seaborn scikit-learn -q

---
## Block 2: Imports

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq
import anndata as ad
import h5py
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import sparse
from scipy.sparse import issparse
from sklearn.neighbors import kneighbors_graph
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.family'] = 'sans-serif'

print('All imports successful.')

---
## Block 3: Upload E-MTAB-13530 Data

Upload the E-MTAB-13530.zip file to Colab, or mount Google Drive if the file is there.

In [ ]:
# Option A: Upload from local machine
# from google.colab import files
# uploaded = files.upload()  # upload E-MTAB-13530.zip

# Option B: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Set path to your zip file
ZIP_PATH = '/content/drive/MyDrive/E-MTAB-13530.zip'  # <-- CHANGE THIS to your actual path

# Or if uploaded directly:
# ZIP_PATH = '/content/E-MTAB-13530.zip'

print(f'Using: {ZIP_PATH}')
print(f'Exists: {os.path.exists(ZIP_PATH)}')

---
## Block 4: Extract Data

In [ ]:
import zipfile

BASE_DIR = '/content/E-MTAB-13530'
os.makedirs(BASE_DIR, exist_ok=True)

# Extract all h5 and spatial tar files
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    members = [m for m in z.namelist() if m.endswith('.h5') or m.endswith('-spatial.tar') or m.endswith('.txt')]
    for m in members:
        z.extract(m, BASE_DIR)

print(f'Extracted {len(members)} files to {BASE_DIR}')

# Extract spatial tars into proper directory structure
import tarfile
tar_files = [f for f in os.listdir(BASE_DIR) if f.endswith('-spatial.tar')]
for tf in tar_files:
    name = tf.replace('-spatial.tar', '')
    spatial_dir = os.path.join(BASE_DIR, f'{name}_spatial', 'spatial')
    os.makedirs(spatial_dir, exist_ok=True)
    with tarfile.open(os.path.join(BASE_DIR, tf), 'r') as t:
        t.extractall(os.path.join(BASE_DIR, f'{name}_spatial', 'spatial'))
    # Fix: files might extract into spatial/spatial — move them up if needed
    inner = os.path.join(spatial_dir, 'spatial')
    if os.path.exists(inner):
        for ff in os.listdir(inner):
            os.rename(os.path.join(inner, ff), os.path.join(spatial_dir, ff))

print(f'Extracted {len(tar_files)} spatial archives')

# Verify one sample
test_dir = os.path.join(BASE_DIR, 'P10_T1_spatial', 'spatial')
if os.path.exists(test_dir):
    print(f'Test directory contents: {os.listdir(test_dir)}')
else:
    # Files extracted flat — restructure
    flat_dir = os.path.join(BASE_DIR, 'P10_T1_spatial')
    if os.path.exists(os.path.join(flat_dir, 'tissue_positions_list.csv')):
        os.makedirs(test_dir, exist_ok=True)
        for ff in ['tissue_positions_list.csv', 'scalefactors_json.json', 'tissue_hires_image.png', 'tissue_lowres_image.png']:
            src = os.path.join(flat_dir, ff)
            if os.path.exists(src):
                os.rename(src, os.path.join(test_dir, ff))
        print(f'Restructured. Contents: {os.listdir(test_dir)}')

---
## Block 5: Helper Functions

In [ ]:
def load_visium_manual(h5_path, spatial_dir):
    """Load Visium data manually, handling duplicate gene names."""
    pos_file = os.path.join(spatial_dir, 'tissue_positions_list.csv')
    
    # Load positions
    pos = pd.read_csv(pos_file, header=None)
    pos.columns = ['barcode', 'in_tissue', 'row', 'col', 'px_y', 'px_x']
    pos = pos[pos['in_tissue'] == 1].copy()
    pos = pos.set_index('barcode')
    
    # Load h5
    with h5py.File(h5_path, 'r') as f:
        grp = f['matrix']
        data = grp['data'][:]
        indices = grp['indices'][:]
        indptr = grp['indptr'][:]
        shape = grp['shape'][:]
        barcodes = [b.decode() for b in grp['barcodes'][:]]
        gene_names = [g.decode() for g in grp['features']['name'][:]]
    
    X = sparse.csc_matrix((data, indices, indptr), shape=shape).T.tocsr()
    
    # Make gene names unique
    counts = Counter(gene_names)
    seen = {}
    unique_genes = []
    for g in gene_names:
        if counts[g] > 1:
            n = seen.get(g, 0)
            unique_genes.append(f'{g}_{n}')
            seen[g] = n + 1
        else:
            unique_genes.append(g)
    
    adata = ad.AnnData(X=X)
    adata.obs_names = barcodes
    adata.var_names = unique_genes
    
    # Filter to in-tissue
    common = [b for b in adata.obs_names if b in pos.index]
    adata = adata[common].copy()
    coords = pos.loc[common, ['px_x', 'px_y']].values.astype(float)
    adata.obsm['spatial'] = coords
    
    return adata


def compute_morans_i(adata, genes, W_mat=None):
    """Compute Moran's I manually using vectorized operations."""
    if W_mat is None:
        W_mat = adata.obsp['spatial_connectivities']
    if issparse(W_mat):
        W_mat = W_mat.toarray()
    
    S0 = W_mat.sum()
    n = W_mat.shape[0]
    morans = []
    
    for gene in genes:
        if gene not in adata.var_names:
            continue
        g_idx = list(adata.var_names).index(gene)
        x = adata.X[:, g_idx]
        if issparse(x):
            x = x.toarray().flatten()
        else:
            x = np.array(x).flatten()
        
        xbar = x.mean()
        xdev = x - xbar
        ss = np.sum(xdev**2)
        if ss == 0:
            continue
        
        # Vectorized: num = xdev^T @ W @ xdev
        num = float(xdev @ W_mat @ xdev)
        I_val = (n / S0) * (num / ss)
        morans.append(I_val)
    
    return morans


def compute_meff(m_i, rho_i):
    """Compute effective sample size from design effect formula."""
    rho = max(0.001, min(rho_i, 0.999))
    DEFF = 1 + (m_i - 1) * rho
    m_eff = m_i / DEFF
    return DEFF, m_eff


print('Helper functions defined.')

---
## Block 6: Process All Sections — Compute $m_i$, $\hat{\rho}_i$, $m_{i,\text{eff}}$

This is the **GO/NO-GO computation**.

In [ ]:
IMMUNE_GENES = ['CD3E', 'CD3D', 'CD8A', 'CD8B', 'CD68', 'CD14', 'PTPRC', 'CD4', 'FOXP3', 'MS4A1']

h5_files = sorted([f for f in os.listdir(BASE_DIR) if f.endswith('-filtered_feature_bc_matrix.h5')])
results = []

for h5_file in h5_files:
    sample_name = h5_file.replace('-filtered_feature_bc_matrix.h5', '')
    spatial_dir = os.path.join(BASE_DIR, f'{sample_name}_spatial', 'spatial')
    h5_path = os.path.join(BASE_DIR, h5_file)
    
    # Check for flat structure (no spatial/ subdirectory)
    if not os.path.exists(os.path.join(spatial_dir, 'tissue_positions_list.csv')):
        flat_dir = os.path.join(BASE_DIR, f'{sample_name}_spatial')
        if os.path.exists(os.path.join(flat_dir, 'tissue_positions_list.csv')):
            os.makedirs(spatial_dir, exist_ok=True)
            for ff in os.listdir(flat_dir):
                if ff != 'spatial' and not os.path.isdir(os.path.join(flat_dir, ff)):
                    src = os.path.join(flat_dir, ff)
                    dst = os.path.join(spatial_dir, ff)
                    if not os.path.exists(dst):
                        os.rename(src, dst)
    
    pos_file = os.path.join(spatial_dir, 'tissue_positions_list.csv')
    if not os.path.exists(pos_file):
        print(f'  SKIP {sample_name}: no positions file')
        continue
    
    try:
        # Load data
        adata = load_visium_manual(h5_path, spatial_dir)
        m_i = adata.n_obs
        
        if m_i < 50:
            print(f'  SKIP {sample_name}: {m_i} spots')
            continue
        
        # Normalize
        sc.pp.normalize_total(adata, target_sum=1e4)
        sc.pp.log1p(adata)
        
        # Build 6-NN graph
        coords = adata.obsm['spatial']
        adj = kneighbors_graph(coords, n_neighbors=6, mode='connectivity')
        adj = (adj + adj.T)
        adj[adj > 1] = 1
        adata.obsp['spatial_connectivities'] = adj
        
        # Find available immune genes
        available = [g for g in IMMUNE_GENES if g in adata.var_names]
        if len(available) == 0:
            sc.pp.highly_variable_genes(adata, n_top_genes=20, flavor='seurat_v3', subset=False)
            available = list(adata.var_names[adata.var['highly_variable']][:5])
        
        # Compute Moran's I
        morans = compute_morans_i(adata, available[:5])
        
        if len(morans) == 0:
            # Fallback: total counts
            x = np.array(adata.X.sum(axis=1)).flatten()
            W_mat = adj.toarray() if issparse(adj) else adj
            xdev = x - x.mean()
            ss = np.sum(xdev**2)
            if ss > 0:
                num = float(xdev @ W_mat @ xdev)
                morans = [(m_i / W_mat.sum()) * (num / ss)]
        
        rho_i = float(np.mean(morans)) if morans else 0.0
        DEFF, m_eff = compute_meff(m_i, rho_i)
        
        # Determine type
        parts = sample_name.split('_')
        is_tumor = len(parts) > 1 and parts[1].startswith('T')
        patient_id = parts[0]
        
        results.append({
            'sample': sample_name,
            'patient': patient_id,
            'type': 'tumor' if is_tumor else 'normal/adjacent',
            'm_i': m_i,
            'rho_i': round(rho_i, 4),
            'DEFF': round(DEFF, 1),
            'm_eff': round(m_eff, 1),
            'genes_used': ', '.join(available[:5]),
        })
        
        print(f'  OK {sample_name}: m={m_i}, rho={rho_i:.4f}, DEFF={DEFF:.1f}, m_eff={m_eff:.1f}')
    
    except Exception as e:
        print(f'  ERROR {sample_name}: {e}')
        continue

df = pd.DataFrame(results)
df = df.sort_values(['patient', 'type', 'sample'])
print(f'\nProcessed {len(df)} sections successfully.')

---
## Block 7: GO / NO-GO Decision

In [ ]:
tumor = df[df['type'] == 'tumor'].copy()
normal = df[df['type'] != 'tumor'].copy()

print('=' * 70)
print('GO / NO-GO DECISION')
print('=' * 70)
print(f'\nTotal sections: {len(df)} ({len(tumor)} tumor, {len(normal)} normal/adjacent)')

print('\n--- TUMOR SECTIONS ---')
print(f'  m_i:   {tumor["m_i"].min()} - {tumor["m_i"].max()} (mean {tumor["m_i"].mean():.0f})')
print(f'  rho_i: {tumor["rho_i"].min():.4f} - {tumor["rho_i"].max():.4f} (mean {tumor["rho_i"].mean():.4f})')
print(f'  DEFF:  {tumor["DEFF"].min():.1f} - {tumor["DEFF"].max():.1f} (mean {tumor["DEFF"].mean():.1f})')
print(f'  m_eff: {tumor["m_eff"].min():.1f} - {tumor["m_eff"].max():.1f} (mean {tumor["m_eff"].mean():.1f})')

print('\n--- NORMAL/ADJACENT SECTIONS ---')
print(f'  m_i:   {normal["m_i"].min()} - {normal["m_i"].max()} (mean {normal["m_i"].mean():.0f})')
print(f'  m_eff: {normal["m_eff"].min():.1f} - {normal["m_eff"].max():.1f} (mean {normal["m_eff"].mean():.1f})')

mean_meff = tumor['m_eff'].mean()
mean_mi = tumor['m_i'].mean()
ratio = mean_meff / mean_mi

print(f'\n>>> HEADLINE: {mean_mi:.0f} spots --> {mean_meff:.1f} effective ({(1-ratio)*100:.1f}% reduction) <<<')

if mean_meff < 20:
    print('\n>>> VERDICT: STRONG EFFECT. PAPER EXISTS. GO. <<<')
elif mean_meff < 100:
    print('\n>>> VERDICT: MODERATE EFFECT. Paper viable. GO. <<<')
else:
    print('\n>>> VERDICT: WEAK EFFECT. Reconsider. <<<')

---
## Block 8: Full Results Table

In [ ]:
# Display full table
display(df[['sample', 'patient', 'type', 'm_i', 'rho_i', 'DEFF', 'm_eff']].style
    .background_gradient(subset=['m_eff'], cmap='RdYlGn')
    .background_gradient(subset=['DEFF'], cmap='YlOrRd')
    .format({'rho_i': '{:.4f}', 'DEFF': '{:.1f}', 'm_eff': '{:.1f}'})
    .set_caption('Effective Sample Size Analysis — E-MTAB-13530 NSCLC')
)

---
## Block 9: FIGURE 1 — The Paper's Central Result

$m_{i,\text{eff}}$ vs $m_i$ with theoretical curves for different $\rho$ values.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- LEFT: m_eff vs m_i with theoretical curves ---
ax = axes[0]

# Theoretical curves
m_range = np.linspace(100, 4000, 500)
for rho_val, color, label in [(0.01, '#2ecc71', r'$\rho = 0.01$'),
                                (0.03, '#3498db', r'$\rho = 0.03$'),
                                (0.05, '#f39c12', r'$\rho = 0.05$'),
                                (0.10, '#e74c3c', r'$\rho = 0.10$')]:
    meff_curve = m_range / (1 + (m_range - 1) * rho_val)
    ax.plot(m_range, meff_curve, color=color, linewidth=1.5, alpha=0.7, label=label)

# Data points
colors = {'tumor': '#e74c3c', 'normal/adjacent': '#3498db'}
for tissue_type in ['tumor', 'normal/adjacent']:
    subset = df[df['type'] == tissue_type]
    ax.scatter(subset['m_i'], subset['m_eff'], c=colors[tissue_type],
              s=60, alpha=0.8, edgecolors='white', linewidth=0.5,
              label=f'{tissue_type} (n={len(subset)})', zorder=5)

ax.set_xlabel('Number of Visium Spots ($m_i$)', fontsize=12)
ax.set_ylabel('Effective Sample Size ($m_{i,eff}$)', fontsize=12)
ax.set_title('Effective Sample Size Collapse\nin NSCLC Spatial Transcriptomics', fontsize=13, fontweight='bold')
ax.legend(fontsize=9, loc='upper left')
ax.set_xlim(0, 4000)
ax.set_ylim(0, 180)
ax.axhline(y=50, color='gray', linestyle='--', alpha=0.3, linewidth=0.8)
ax.text(3800, 52, '$m_{eff}=50$', fontsize=8, color='gray', ha='right')

# --- RIGHT: DEFF distribution ---
ax = axes[1]

# Bar chart: tumor vs normal
x_pos = np.arange(len(df))
colors_bar = [colors[t] for t in df['type']]
bars = ax.bar(x_pos, df['DEFF'].values, color=colors_bar, alpha=0.8, edgecolor='white', linewidth=0.3)
ax.set_xlabel('Tissue Section (sorted by patient)', fontsize=12)
ax.set_ylabel('Design Effect (DEFF)', fontsize=12)
ax.set_title('Design Effect per Section\n(higher = more redundant spots)', fontsize=13, fontweight='bold')
ax.set_xticks(x_pos[::4])
ax.set_xticklabels(df['sample'].values[::4], rotation=45, ha='right', fontsize=7)

# Add legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#e74c3c', label='Tumor'),
                   Patch(facecolor='#3498db', label='Normal/Adjacent')]
ax.legend(handles=legend_elements, fontsize=9)

plt.tight_layout()
plt.savefig('/content/Figure1_meff.png', dpi=300, bbox_inches='tight')
plt.savefig('/content/Figure1_meff.pdf', bbox_inches='tight')
plt.show()

print('\nFigure 1 saved to /content/Figure1_meff.png and .pdf')

---
## Block 10: Attenuation Bias Estimation

Estimate how much the hazard ratio would be attenuated given the observed $m_{i,\text{eff}}$ values.

In [ ]:
# Theory-based variance model (zero free parameters)
# sigma_i^2 = W_i(1-W_i) * DEFF_i / m_i
# For this demo, use mean W = 0.35 (typical immune enrichment)

W_typical = 0.35  # typical immune enrichment proportion
sigma_X2 = 0.04   # assumed between-patient signal variance

print('ATTENUATION BIAS ESTIMATION')
print('=' * 60)
print(f'Assumed W ~ {W_typical}, sigma_X^2 = {sigma_X2}')
print()

tumor_data = tumor.copy()

# Compute sigma_i^2 for each tumor section
tumor_data['sigma_i2'] = W_typical * (1 - W_typical) * tumor_data['DEFF'] / tumor_data['m_i']

# Patient-specific reliability ratio
tumor_data['lambda_i'] = sigma_X2 / (sigma_X2 + tumor_data['sigma_i2'])

# Global reliability ratio (standard RC)
mean_sigma_U2 = tumor_data['sigma_i2'].mean()
lambda_global = sigma_X2 / (sigma_X2 + mean_sigma_U2)

# Attenuation
true_beta = -1.5  # assumed true log-HR
naive_beta = lambda_global * true_beta

print(f'Mean sigma_i^2 (tumor):  {mean_sigma_U2:.4f}')
print(f'Global lambda:           {lambda_global:.3f}')
print(f'\nTrue beta:               {true_beta:.2f}  (HR = {np.exp(true_beta):.3f})')
print(f'Naive beta (attenuated): {naive_beta:.2f}  (HR = {np.exp(naive_beta):.3f})')
print(f'Attenuation:             {(1 - lambda_global)*100:.1f}%')

print(f'\n--- Patient-specific lambda_i ---')
print(tumor_data[['sample', 'm_i', 'rho_i', 'DEFF', 'm_eff', 'sigma_i2', 'lambda_i']].to_string(index=False))

print(f'\nlambda_i range: {tumor_data["lambda_i"].min():.3f} - {tumor_data["lambda_i"].max():.3f}')
print(f'lambda_i spread: {(tumor_data["lambda_i"].max() - tumor_data["lambda_i"].min()):.3f}')
print(f'\nThis {(tumor_data["lambda_i"].max() - tumor_data["lambda_i"].min()):.3f} spread is WHY patient-specific lambda_i matters.')
print('Standard RC with one lambda ignores this variation.')

---
## Block 11: FIGURE 2 — Attenuation Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- LEFT: lambda_i distribution ---
ax = axes[0]
ax.barh(range(len(tumor_data)), tumor_data['lambda_i'].values, 
        color=['#e74c3c' if l < 0.8 else '#f39c12' if l < 0.9 else '#2ecc71' 
               for l in tumor_data['lambda_i']], alpha=0.8)
ax.axvline(x=lambda_global, color='black', linestyle='--', linewidth=2, label=f'Global $\\lambda$ = {lambda_global:.3f}')
ax.set_yticks(range(len(tumor_data)))
ax.set_yticklabels(tumor_data['sample'].values, fontsize=8)
ax.set_xlabel('Reliability Ratio $\\lambda_i$', fontsize=11)
ax.set_title('Patient-Specific\nReliability', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.set_xlim(0, 1)

# --- MIDDLE: HR attenuation ---
ax = axes[1]
true_hr = np.exp(true_beta)
beta_range = np.linspace(-2.5, 0, 100)
for lam, color, label in [(lambda_global, '#e74c3c', f'Naive ($\\lambda$={lambda_global:.2f})'),
                           (0.90, '#f39c12', '$\\lambda$=0.90'),
                           (1.00, '#2ecc71', 'Oracle ($\\lambda$=1.00)')]:
    ax.plot(np.exp(beta_range), np.exp(lam * beta_range), color=color, linewidth=2, label=label)

ax.axhline(y=1, color='gray', linestyle=':', alpha=0.5)
ax.axvline(x=true_hr, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel('True HR', fontsize=11)
ax.set_ylabel('Estimated HR', fontsize=11)
ax.set_title('Attenuation of\nHazard Ratio', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.set_xlim(0.05, 1.05)
ax.set_ylim(0.05, 1.05)
ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, linewidth=0.5)

# --- RIGHT: Tumor vs Normal rho ---
ax = axes[2]
data_plot = pd.concat([
    tumor[['rho_i']].assign(type='Tumor'),
    normal[['rho_i']].assign(type='Normal/Adjacent')
])
sns.boxplot(data=data_plot, x='type', y='rho_i', ax=ax, 
            palette={'Tumor': '#e74c3c', 'Normal/Adjacent': '#3498db'})
sns.stripplot(data=data_plot, x='type', y='rho_i', ax=ax, color='black', alpha=0.5, size=4)
ax.set_ylabel("Moran's I ($\\hat{\\rho}_i$)", fontsize=11)
ax.set_xlabel('')
ax.set_title('Spatial Autocorrelation:\nTumor > Normal', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('/content/Figure2_attenuation.png', dpi=300, bbox_inches='tight')
plt.show()

print('Figure 2 saved.')

---
## Block 12: GRC Correction Demonstration

Demonstrate how patient-specific $\lambda_i$ corrects differently than global $\lambda$.

In [ ]:
# Simulate observed W_i for tumor sections
np.random.seed(42)

n_patients = len(tumor_data)
mu_X = 0.35

# Generate "true" X_i
X_true = np.random.normal(mu_X, np.sqrt(sigma_X2), n_patients)
X_true = np.clip(X_true, 0.05, 0.95)

# Generate errors with patient-specific variance
sigma_i2_vals = tumor_data['sigma_i2'].values
U_i = np.array([np.random.normal(0, np.sqrt(s)) for s in sigma_i2_vals])
W_obs = X_true + U_i
W_obs = np.clip(W_obs, 0, 1)

# Standard RC (one lambda)
mu_W = W_obs.mean()
X_rc = mu_W + lambda_global * (W_obs - mu_W)

# GRC (patient-specific lambda_i)
lambda_i_vals = tumor_data['lambda_i'].values
X_grc = mu_W + lambda_i_vals * (W_obs - mu_W)

# Compare errors
err_naive = np.mean((W_obs - X_true)**2)
err_rc = np.mean((X_rc - X_true)**2)
err_grc = np.mean((X_grc - X_true)**2)

print('GRC CORRECTION DEMONSTRATION')
print('=' * 60)
print(f'MSE (Naive W_i):     {err_naive:.5f}')
print(f'MSE (Standard RC):   {err_rc:.5f}  ({(1-err_rc/err_naive)*100:.1f}% improvement)')
print(f'MSE (GRC, lambda_i): {err_grc:.5f}  ({(1-err_grc/err_naive)*100:.1f}% improvement)')
print(f'\nGRC additional improvement over RC: {(1-err_grc/err_rc)*100:.1f}%')

# Show per-patient comparison
comp = pd.DataFrame({
    'sample': tumor_data['sample'].values,
    'X_true': np.round(X_true, 3),
    'W_obs': np.round(W_obs, 3),
    'X_rc': np.round(X_rc, 3),
    'X_grc': np.round(X_grc, 3),
    'err_naive': np.round(np.abs(W_obs - X_true), 3),
    'err_rc': np.round(np.abs(X_rc - X_true), 3),
    'err_grc': np.round(np.abs(X_grc - X_true), 3),
})
print(f'\n--- Per-Section Comparison ---')
print(comp.to_string(index=False))

---
## Block 13: FIGURE 3 — GRC vs Standard RC

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (x_val, label, color) in zip(axes, 
    [(W_obs, 'Naive $W_i$', '#e74c3c'),
     (X_rc, 'Standard RC', '#f39c12'),
     (X_grc, 'GRC ($\\lambda_i$)', '#2ecc71')]):
    ax.scatter(X_true, x_val, c=color, s=60, alpha=0.7, edgecolors='white')
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.3)
    mse = np.mean((x_val - X_true)**2)
    corr = np.corrcoef(X_true, x_val)[0, 1]
    ax.set_xlabel('True $X_i$', fontsize=11)
    ax.set_ylabel(f'Estimated ({label})', fontsize=11)
    ax.set_title(f'{label}\nMSE={mse:.4f}, r={corr:.3f}', fontsize=12, fontweight='bold')
    ax.set_xlim(0, 0.7)
    ax.set_ylim(0, 0.7)

plt.tight_layout()
plt.savefig('/content/Figure3_GRC_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print('Figure 3 saved.')

---
## Block 14: Graph Sensitivity Analysis

Compute $m_{i,\text{eff}}$ under different graph types (4-NN, 6-NN, 8-NN).

In [ ]:
# Use first tumor sample for graph sensitivity
sample_name = tumor_data.iloc[0]['sample']
h5_path = os.path.join(BASE_DIR, f'{sample_name}-filtered_feature_bc_matrix.h5')
spatial_dir = os.path.join(BASE_DIR, f'{sample_name}_spatial', 'spatial')

adata_test = load_visium_manual(h5_path, spatial_dir)
sc.pp.normalize_total(adata_test, target_sum=1e4)
sc.pp.log1p(adata_test)
coords_test = adata_test.obsm['spatial']
m_test = adata_test.n_obs

available_test = [g for g in IMMUNE_GENES if g in adata_test.var_names][:5]

graph_results = []
for k in [4, 6, 8, 10, 15, 20]:
    adj_k = kneighbors_graph(coords_test, n_neighbors=k, mode='connectivity')
    adj_k = (adj_k + adj_k.T)
    adj_k[adj_k > 1] = 1
    adata_test.obsp['spatial_connectivities'] = adj_k
    
    morans_k = compute_morans_i(adata_test, available_test)
    rho_k = np.mean(morans_k) if morans_k else 0
    DEFF_k, meff_k = compute_meff(m_test, rho_k)
    
    graph_results.append({'k': k, 'rho': rho_k, 'DEFF': DEFF_k, 'm_eff': meff_k})
    print(f'  k={k:2d}-NN: rho={rho_k:.4f}, DEFF={DEFF_k:.1f}, m_eff={meff_k:.1f}')

gdf = pd.DataFrame(graph_results)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(gdf['k'], gdf['m_eff'], 'o-', color='#e74c3c', linewidth=2, markersize=8)
ax.set_xlabel('Number of Neighbors (k)', fontsize=12)
ax.set_ylabel('$m_{i,eff}$', fontsize=12)
ax.set_title(f'Graph Sensitivity: {sample_name} (m={m_test})', fontsize=13, fontweight='bold')
ax.axhline(y=50, color='gray', linestyle='--', alpha=0.3)
for _, row in gdf.iterrows():
    ax.annotate(f'{row["m_eff"]:.0f}', (row['k'], row['m_eff']), 
                textcoords='offset points', xytext=(0, 10), ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('/content/Figure4_graph_sensitivity.png', dpi=300, bbox_inches='tight')
plt.show()
print('Figure 4 saved.')

---
## Block 15: Save All Results

In [ ]:
# Save results CSV
df.to_csv('/content/meff_results.csv', index=False)
print('Results saved to /content/meff_results.csv')

# Download results
from google.colab import files
files.download('/content/meff_results.csv')
files.download('/content/Figure1_meff.png')
files.download('/content/Figure2_attenuation.png')
files.download('/content/Figure3_GRC_comparison.png')
files.download('/content/Figure4_graph_sensitivity.png')

---
## Block 16: Summary for Paper

**Key results from E-MTAB-13530 NSCLC Visium data:**

In [ ]:
print('=' * 70)
print('PAPER 2 — KEY RESULTS SUMMARY')
print('=' * 70)
print(f'''
Dataset: E-MTAB-13530 (NSCLC, 10x Visium)
Patients: {df["patient"].nunique()} ({len(tumor)} tumor sections, {len(normal)} normal)

HEADLINE RESULT:
  {tumor["m_i"].mean():.0f} Visium spots --> {tumor["m_eff"].mean():.1f} effective observations
  {(1 - tumor["m_eff"].mean()/tumor["m_i"].mean())*100:.1f}% reduction in effective sample size

DESIGN EFFECT:
  Tumor DEFF range:  {tumor["DEFF"].min():.1f} - {tumor["DEFF"].max():.1f}
  Tumor DEFF mean:   {tumor["DEFF"].mean():.1f}

SPATIAL AUTOCORRELATION:
  Tumor rho mean:    {tumor["rho_i"].mean():.4f}
  Normal rho mean:   {normal["rho_i"].mean():.4f}
  Tumor > Normal:    YES (more spatially structured)

ATTENUATION (estimated):
  Global lambda:     {lambda_global:.3f}
  Attenuation:       {(1-lambda_global)*100:.1f}%
  If true HR = 0.22, naive HR ≈ {np.exp(lambda_global * true_beta):.3f}

ABSTRACT SENTENCE:
  "Spatial autocorrelation in NSCLC tumor tissue reduces the effective
   sample size of Visium-derived biomarkers from {tumor["m_i"].mean():.0f} spots
   to {tumor["m_eff"].mean():.1f} independent observations ({(1-tumor["m_eff"].mean()/tumor["m_i"].mean())*100:.0f}% reduction)."
''')